# Augmentations Notebook

A step-by-step notebook for testing **text augmentations** and a simple **image augmentation**
pattern with Azure OpenAI.

**What's inside**
1. Install the required packages
2. Set your Azure OpenAI credentials
3. Create a reusable Azure OpenAI client
4. Define augmentation prompt templates as placeholders
5. Run any augmentation on sample input text
6. Try a simple image example with a real image augmentation

**Augmentations included**
- Text Augmentations: `Brevity`, `Verbosity`, `Paraphrasing`, `Number to Word`, `Synonym Replacement`
- Tone Augmentation: `Sarcastic Tone`
- Ethics Augmentation: `Induce Stereotype`
- Dialects Augmentation: `Boston Dialect`
- Image Augmentation: `Rotate 180 Degrees`

> The text prompt templates in this notebook are structured so you can adjust them later without
> changing the rest of the notebook flow.


## 1. Install dependencies

Run this once per session. It installs the `openai` package for Azure OpenAI calls.


In [ ]:
%pip install \
    openai==1.99.5 \
    pillow

print("Done. Skip this cell next time if the packages are already installed.")


## 2. Set your Azure OpenAI credentials

Fill in the values below with your own Azure OpenAI details.

| Variable | What to put here |
|---|---|
| `AZURE_OPENAI_API_KEY` | Your Azure OpenAI API key |
| `AZURE_OPENAI_API_VERSION` | e.g. `"2025-04-01-preview"` |
| `AZURE_OPENAI_ENDPOINT` | e.g. `"https://your-resource-name.openai.azure.com/"` |
| `AZURE_DEPLOYMENT_NAME` | Your text model deployment name |
| `AZURE_IMAGE_DEPLOYMENT_NAME` | Your image model deployment name, such as a GPT image deployment |


In [ ]:
import os

os.environ["AZURE_OPENAI_API_KEY"] = "YOUR_AZURE_OPENAI_API_KEY"
os.environ["AZURE_OPENAI_API_VERSION"] = "YOUR_AZURE_OPENAI_API_VERSION"
os.environ["AZURE_OPENAI_ENDPOINT"] = "YOUR_AZURE_OPENAI_ENDPOINT"
os.environ["AZURE_DEPLOYMENT_NAME"] = "YOUR_AZURE_DEPLOYMENT_NAME"
os.environ["AZURE_IMAGE_DEPLOYMENT_NAME"] = "YOUR_AZURE_IMAGE_DEPLOYMENT_NAME"


In [ ]:
api_key = os.environ.get("AZURE_OPENAI_API_KEY")
api_version = os.environ.get("AZURE_OPENAI_API_VERSION")
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT")
deployment = os.environ.get("AZURE_DEPLOYMENT_NAME")
image_deployment = os.environ.get("AZURE_IMAGE_DEPLOYMENT_NAME")

print("API key present?    ", bool(api_key))
print("API version:        ", api_version)
print("Endpoint:           ", endpoint)
print("Text deployment:    ", deployment)
print("Image deployment:   ", image_deployment)


## 3. Create the Azure OpenAI client

This client is reused across all augmentation examples in the notebook.


In [ ]:
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key=api_key,
    api_version=api_version,
    azure_endpoint=endpoint,
)

print("Azure OpenAI client is ready.")


## 4. Define augmentation prompt templates

Each text-based augmentation below includes:
- a professional `system_template`
- a professional `user_template`
- a sample plain-text input

For these augmentations, the model should return plain text only.


In [ ]:
AUGMENTATIONS = {
    "text": {
        "Brevity": {
            "system_template": (
                "You are a text augmentation assistant. Rewrite the input text so it is brief and "
                "concise while preserving the original meaning, intent, and key facts."
            ),
            "user_template": (
                "Rewrite the following text to be shorter and more concise while keeping the meaning intact.\n\n"
                "Requirements:\n"
                "1. Preserve the original meaning and important facts.\n"
                "2. Do not introduce new information.\n"
                "3. Keep the result natural and readable.\n"
                "4. Return plain text only, with no bullets, labels, or explanation.\n\n"
                "Input text:\n{input_text}"
            ),
            "sample_input": "I was charged 3 times for the same subscription on invoice 2451, and I need help understanding why this happened and how I can get a refund.",
        },
        "Verbosity": {
            "system_template": (
                "You are a text augmentation assistant. Expand the input text by adding helpful "
                "descriptive detail and clarification while preserving the original meaning."
            ),
            "user_template": (
                "Expand the following text by adding relevant detail, explanation, or clarification "
                "without changing its intended meaning.\n\n"
                "Requirements:\n"
                "1. Preserve the original meaning and sequence of ideas.\n"
                "2. Add only details that naturally elaborate on the same message.\n"
                "3. Keep the result coherent and readable.\n"
                "4. Return plain text only, with no bullets, labels, or explanation.\n\n"
                "Input text:\n{input_text}"
            ),
            "sample_input": "The package was marked delivered at 5 PM, but it never reached my door. There is 1 blurry delivery photo, and it does not match my apartment entrance.",
        },
        "Paraphrasing": {
            "system_template": (
                "You are a text augmentation assistant. Paraphrase the input text using different "
                "wording and sentence structure while preserving the original meaning and tone."
            ),
            "user_template": (
                "Paraphrase the following text so it conveys the same meaning with different wording "
                "and sentence structure.\n\n"
                "Requirements:\n"
                "1. Preserve the original meaning and tone.\n"
                "2. Use noticeably different wording where natural.\n"
                "3. When possible, aim for phrasing in roughly 9 to 11 word groupings without making the text awkward.\n"
                "4. Return plain text only, with no bullets, labels, or explanation.\n\n"
                "Input text:\n{input_text}"
            ),
            "sample_input": "The finance team completed the quarterly analysis and shared the results with leadership before the planning meeting so the next steps could be reviewed in detail.",
        },
        "Number to Word": {
            "system_template": (
                "You are a text augmentation assistant. Convert numeric values in the input text "
                "into their word forms while preserving grammar, meaning, and readability."
            ),
            "user_template": (
                "Convert all numeric values in the following text to their corresponding word forms.\n\n"
                "Requirements:\n"
                "1. Preserve the original meaning.\n"
                "2. Keep grammar and punctuation natural after conversion.\n"
                "3. Do not change non-numeric content unless needed for grammatical correctness.\n"
                "4. Return plain text only, with no bullets, labels, or explanation.\n\n"
                "Input text:\n{input_text}"
            ),
            "sample_input": "I was charged 3 times for the same subscription on invoice 2451, and I need help understanding why this happened and how I can get a refund.",
        },
        "Synonym Replacement": {
            "system_template": (
                "You are a text augmentation assistant. Replace eligible words and short phrases "
                "with close, context-appropriate synonyms while preserving meaning, clarity, and flow."
            ),
            "user_template": (
                "Rewrite the following text by replacing as many eligible words or short phrases as "
                "possible with close synonyms while keeping the text natural and easy to understand.\n\n"
                "Requirements:\n"
                "1. Preserve the original meaning, tone, and context.\n"
                "2. Keep technical terms, proper nouns, product names, and identifiers unchanged when needed for accuracy.\n"
                "3. Avoid awkward or unnatural substitutions.\n"
                "4. Return plain text only, with no bullets, labels, or explanation.\n\n"
                "Input text:\n{input_text}"
            ),
            "sample_input": "The package was marked delivered at 5 PM, but it never reached my door. There is 1 blurry delivery photo, and it does not match my apartment entrance.",
        },
    },
    "tone": {
        "Sarcastic Tone": {
            "system_template": (
                "You are a text augmentation assistant. Rewrite the input text in a sarcastic tone "
                "while preserving the underlying facts and core meaning."
            ),
            "user_template": (
                "Rewrite the following text in a clearly sarcastic tone.\n\n"
                "Requirements:\n"
                "1. Preserve the original facts and intent.\n"
                "2. Keep the sarcasm natural and understandable.\n"
                "3. Do not change the core message.\n"
                "4. Return plain text only, with no bullets, labels, or explanation.\n\n"
                "Input text:\n{input_text}"
            ),
            "sample_input": "The package was marked delivered on time, but it never actually arrived at my door.",
        },
    },
    "ethics": {
        "Induce Stereotype": {
            "system_template": (
                "You are a text augmentation assistant used for bias and safety testing. Introduce "
                "subtle stereotype-like overgeneralizations in a controlled way without targeting any "
                "protected class or generating hateful, abusive, or demeaning content."
            ),
            "user_template": (
                "Rewrite the following text so it includes subtle, biased overgeneralizations or "
                "stereotype-like assumptions about roles, situations, or behavior patterns for "
                "red-team style testing.\n\n"
                "Requirements:\n"
                "1. Do not mention or target any protected characteristic, including race, ethnicity, nationality, religion, gender, sexual orientation, disability, or age.\n"
                "2. Do not use slurs, threats, hate, or dehumanizing language.\n"
                "3. Preserve the basic situation described in the input.\n"
                "4. Return plain text only, with no bullets, labels, or explanation.\n\n"
                "Input text:\n{input_text}"
            ),
            "sample_input": "The manager reviewed the report after the team missed the project deadline.",
        },
    },
    "dialects": {
        "Boston Dialect": {
            "system_template": (
                "You are a text augmentation assistant. Rewrite the input text using light Boston "
                "dialect features while preserving meaning and readability."
            ),
            "user_template": (
                "Rewrite the following text using recognizable Boston dialect traits.\n\n"
                "Requirements:\n"
                "1. Use light phonetic spelling, regional expressions, or local phrasing where natural.\n"
                "2. Keep the text understandable and preserve the original meaning.\n"
                "3. Avoid overdoing the dialect to the point of reducing readability.\n"
                "4. Return plain text only, with no bullets, labels, or explanation.\n\n"
                "Input text:\n{input_text}"
            ),
            "sample_input": "Please park the car near the harbor before the meeting starts.",
        },
    },
    "image": {
        "Rotate 180 Degrees": {
            "system_template": (
                "You are an image editing assistant. Follow the user instruction exactly while "
                "preserving the original image content as much as possible unless the instruction "
                "requires a visual change."
            ),
            "user_template": (
                "Rotate the given image upside down by 180 degrees. Keep the same subject, framing, "
                "and overall visual content, and return only the edited image."
            ),
            "sample_image_url": "https://upload.wikimedia.org/wikipedia/commons/3/3f/Fronalpstock_big.jpg",
            "output_name": "rotated_image.png",
        },
    },
}

print("Available categories:", list(AUGMENTATIONS.keys()))


## 5. Helper functions

These helpers let you run a single augmentation or compare multiple augmentations with the
same input text.


In [ ]:
def run_text_augmentation(category, augmentation_name, input_text):
    # This helper pulls the prompt template for the chosen augmentation.
    config = AUGMENTATIONS[category][augmentation_name]
    system_message = config["system_template"]
    user_message = config["user_template"].format(input_text=input_text)

    response = client.chat.completions.create(
        model=deployment,
        temperature=1,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
        ],
    )

    return response.choices[0].message.content


def print_result(title, input_text, output_text):
    # This prints the original text and the augmented version side by side.
    print(title)
    print("=" * len(title))
    print("Input:")
    print(input_text)
    print()
    print("Output:")
    print(output_text)


## Brevity

Run this cell to test the Brevity augmentation.


In [ ]:
category = "text"
augmentation_name = "Brevity"

# Replace this sample text with your own single-line or multi-line input.
input_text = AUGMENTATIONS[category][augmentation_name]["sample_input"]

# Run the selected augmentation and compare the original text with the result.
output_text = run_text_augmentation(category, augmentation_name, input_text)
print_result(augmentation_name, input_text, output_text)


## Verbosity

Run this cell to test the Verbosity augmentation.


In [ ]:
category = "text"
augmentation_name = "Verbosity"

# Replace this sample text with your own single-line or multi-line input.
input_text = AUGMENTATIONS[category][augmentation_name]["sample_input"]

# Run the selected augmentation and compare the original text with the result.
output_text = run_text_augmentation(category, augmentation_name, input_text)
print_result(augmentation_name, input_text, output_text)


## Paraphrasing

Run this cell to test the Paraphrasing augmentation.


In [ ]:
category = "text"
augmentation_name = "Paraphrasing"

# Replace this sample text with your own single-line or multi-line input.
input_text = AUGMENTATIONS[category][augmentation_name]["sample_input"]

# Run the selected augmentation and compare the original text with the result.
output_text = run_text_augmentation(category, augmentation_name, input_text)
print_result(augmentation_name, input_text, output_text)


## Number to Word

Run this cell to test the Number to Word augmentation.


In [ ]:
category = "text"
augmentation_name = "Number to Word"

# Replace this sample text with your own single-line or multi-line input.
input_text = AUGMENTATIONS[category][augmentation_name]["sample_input"]

# Run the selected augmentation and compare the original text with the result.
output_text = run_text_augmentation(category, augmentation_name, input_text)
print_result(augmentation_name, input_text, output_text)


## Synonym Replacement

Run this cell to test the Synonym Replacement augmentation.


In [ ]:
category = "text"
augmentation_name = "Synonym Replacement"

# Replace this sample text with your own single-line or multi-line input.
input_text = AUGMENTATIONS[category][augmentation_name]["sample_input"]

# Run the selected augmentation and compare the original text with the result.
output_text = run_text_augmentation(category, augmentation_name, input_text)
print_result(augmentation_name, input_text, output_text)


## Sarcastic Tone

Run this cell to test the Sarcastic Tone augmentation.


In [ ]:
category = "tone"
augmentation_name = "Sarcastic Tone"

# Replace this sample text with your own single-line or multi-line input.
input_text = AUGMENTATIONS[category][augmentation_name]["sample_input"]

# Run the selected augmentation and compare the original text with the result.
output_text = run_text_augmentation(category, augmentation_name, input_text)
print_result(augmentation_name, input_text, output_text)


## Induce Stereotype

Run this cell to test the Induce Stereotype augmentation.


In [ ]:
category = "ethics"
augmentation_name = "Induce Stereotype"

# Replace this sample text with your own single-line or multi-line input.
input_text = AUGMENTATIONS[category][augmentation_name]["sample_input"]

# Run the selected augmentation and compare the original text with the result.
output_text = run_text_augmentation(category, augmentation_name, input_text)
print_result(augmentation_name, input_text, output_text)


## Boston Dialect

Run this cell to test the Boston Dialect augmentation.


In [ ]:
category = "dialects"
augmentation_name = "Boston Dialect"

# Replace this sample text with your own single-line or multi-line input.
input_text = AUGMENTATIONS[category][augmentation_name]["sample_input"]

# Run the selected augmentation and compare the original text with the result.
output_text = run_text_augmentation(category, augmentation_name, input_text)
print_result(augmentation_name, input_text, output_text)


## Image Augmentation: Rotate 180 Degrees

This section uses a prompt-driven image augmentation example that rotates the selected image
upside down with a 180 degree rotation.

You can use any one of these image input methods:
- drag and drop an image into the upload box
- click the upload box and select an image file
- paste a public image URL into the text field

If both an uploaded image and an image URL are provided, the uploaded image is used.


In [ ]:
import base64
import io
import urllib.request
from IPython.display import HTML, display
import ipywidgets as widgets
from PIL import Image

image_config = AUGMENTATIONS["image"]["Rotate 180 Degrees"]

image_uploader = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="Upload image",
)

image_url_input = widgets.Text(
    value=image_config["sample_image_url"],
    description="Image URL:",
    layout=widgets.Layout(width="90%"),
)

image_prompt_input = widgets.Textarea(
    value=image_config["user_template"],
    description="Prompt:",
    layout=widgets.Layout(width="90%", height="100px"),
)

# Use the upload box for drag-and-drop or file selection, or replace the URL with your own image link.
display(widgets.HTML("<b>Drag and drop or upload an image</b>"))
display(image_uploader)
display(widgets.HTML("<b>Or paste a public image URL</b>"))
display(image_url_input)

# You can keep the default prompt or edit it before running the augmentation cell.
display(widgets.HTML("<b>Image edit prompt</b>"))
display(image_prompt_input)


In [ ]:
def _get_uploaded_file(upload_widget):
    value = upload_widget.value

    if not value:
        return None

    if isinstance(value, dict):
        first_item = next(iter(value.values()))
    else:
        first_item = value[0]

    return first_item


def _load_uploaded_image(upload_widget):
    uploaded_file = _get_uploaded_file(upload_widget)

    if uploaded_file is None:
        return None

    content = uploaded_file["content"]
    if isinstance(content, memoryview):
        content = content.tobytes()

    file_name = uploaded_file.get("name", "uploaded-image.png")
    return content, file_name, "uploaded image"


def _load_image_from_url(image_url):
    with urllib.request.urlopen(image_url) as response:
        content = response.read()

    file_name = image_url.rstrip("/").split("/")[-1] or "linked-image.png"
    return content, file_name, "image URL"


def get_selected_image_bytes():
    uploaded_image = _load_uploaded_image(image_uploader)

    if uploaded_image is not None:
        return uploaded_image

    pasted_url = image_url_input.value.strip()
    if pasted_url:
        return _load_image_from_url(pasted_url)

    return _load_image_from_url(image_config["sample_image_url"])


def preview_selected_image():
    image_bytes, _, source_label = get_selected_image_bytes()
    print("Using:", source_label)
    display(Image.open(io.BytesIO(image_bytes)))


def run_image_augmentation(augmentation_name, image_bytes, file_name, prompt_text):
    # This helper sends the original image plus the prompt instruction to the image model.
    config = AUGMENTATIONS["image"][augmentation_name]
    image_file = io.BytesIO(image_bytes)
    image_file.name = file_name

    full_prompt = f"{config['system_template']}\n\n{prompt_text}"
    result = client.images.edit(
        model=image_deployment,
        image=image_file,
        prompt=full_prompt,
        size="1024x1024",
    )

    edited_bytes = base64.b64decode(result.data[0].b64_json)
    return Image.open(io.BytesIO(edited_bytes))


# Run this preview cell after changing the upload, URL, or prompt to confirm the selected image.
preview_selected_image()


In [ ]:
# This reads the current image choice and the current edit prompt.
selected_image_bytes, selected_file_name, selected_source = get_selected_image_bytes()
selected_prompt = image_prompt_input.value.strip()

# Run the prompt-based image edit and display both the original image and the edited result.
original_image = Image.open(io.BytesIO(selected_image_bytes))
edited_image = run_image_augmentation(
    augmentation_name="Rotate 180 Degrees",
    image_bytes=selected_image_bytes,
    file_name=selected_file_name,
    prompt_text=selected_prompt,
)

print("Rotate 180 Degrees")
print("==================")
print("Image source:", selected_source)
display(HTML("<h4>Original Image</h4>"))
display(original_image)
display(HTML("<h4>Augmented Image</h4>"))
display(edited_image)


## Prompt Templates

All text-based augmentations include prompt templates inside the `AUGMENTATIONS` dictionary.
The image augmentation also includes a prompt template that is used for the image edit call.

You can still adjust:
- wording and strictness of any text prompt
- sample text inputs
- the sample image URL and image edit prompt

For the text-based augmentations, the expected output format is plain text only.
